# 05 · Demo — the web app


> **Research prototype — not a medical device.** Nothing produced by these notebooks may be
> used to diagnose, treat, or make any decision about a patient.


Runs the FastAPI service and the single-page frontend from inside Colab, exposed
through a Cloudflare quick tunnel.

The page has three panels beyond the predictions themselves:

- **Grad-CAM** — click any finding to see the heatmap for *that* finding. All four
  come back from one forward pass, so switching is instant.
- **Diffusion** — generate a film from any label combination with live guidance and
  DDIM-step controls, and SDEdit-refine your own upload. Needs `diffusion.pt` in
  the bundle (notebook 02, then re-run `dvlhg eval`).
- **Vision-language** — image-text alignment per finding, and the same film re-run
  with the report blanked so you can read off what the language channel adds.

**Anyone with that URL can reach your model.** It is a public tunnel with no
authentication. Do not upload identifiable patient data, and stop the tunnel
when you are finished.

In [ ]:
# --- 1. where results live -------------------------------------------------
# Mounting Drive is strongly recommended: Colab disconnects, and every stage
# here writes a resumable checkpoint. Without Drive you start over.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = '/content/drive/MyDrive/dvlhg'
else:
    RUN_ROOT = '/content/dvlhg'

# --- 2. get the code -------------------------------------------------------
# Pick ONE. 'clone' is easiest once you have pushed this repo to GitHub.
SOURCE = 'clone'        # 'clone' | 'zip' | 'drive'
REPO_URL = 'https://github.com/abelsangeeth/DVL-Hyperparameter-for-Lung-Disease-Diagnosis.git'
ZIP_PATH = '/content/dvl-hypergraph.zip'          # if SOURCE == 'zip'
DRIVE_CODE = '/content/drive/MyDrive/dvl-hypergraph'  # if SOURCE == 'drive'

import os, shutil, subprocess, sys
CODE = '/content/dvl-hypergraph'
if not os.path.exists(CODE):
    if SOURCE == 'clone':
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, CODE], check=True)
    elif SOURCE == 'zip':
        if not os.path.exists(ZIP_PATH):
            from google.colab import files
            up = files.upload()            # choose the zip from scripts/make_colab_zip.py
            ZIP_PATH = '/content/' + next(iter(up))
        shutil.unpack_archive(ZIP_PATH, '/content/')
    elif SOURCE == 'drive':
        shutil.copytree(DRIVE_CODE, CODE)
print('code at', CODE, '| contents:', sorted(os.listdir(CODE))[:8])

# --- 3. dependencies -------------------------------------------------------
# Colab already ships torch/torchvision built for its CUDA - never reinstall them.
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'open_clip_torch>=2.24', 'timm>=0.9.12', 'transformers>=4.35',
                'fastapi', 'uvicorn', 'python-multipart'], check=True)

sys.path.insert(0, os.path.join(CODE, 'src'))
os.chdir(CODE)
os.environ['PYTHONPATH'] = os.path.join(CODE, 'src')
os.environ['RUN_ROOT'] = RUN_ROOT
print('run root ->', RUN_ROOT)

### Point at the bundle

In [ ]:
import os
BUNDLE = os.path.join(os.environ['RUN_ROOT'], 'export')
print('bundle:', BUNDLE, '|', sorted(os.listdir(BUNDLE)))

### Start the API

In [ ]:
import os, subprocess, sys, time

env = dict(os.environ,
           DVLHG_BUNDLE=BUNDLE, DVLHG_DEVICE='auto', DVLHG_NEIGHBOURS='6',
           PYTHONPATH=os.path.join(os.getcwd(), 'src'))
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'dvlhg.serve.api:app', '--host', '0.0.0.0', '--port', '8000'],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

import urllib.request
for attempt in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2)
        print('server up'); break
    except Exception:
        time.sleep(1)
else:
    print(server.stdout.read()[-3000:])

### Open a public tunnel

In [ ]:
import re, subprocess, time
subprocess.run('wget -q -O /usr/local/bin/cloudflared '
               'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
               ' && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
for line in tunnel.stdout:
    match = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0); break
print('\n  OPEN THIS:', url, '\n')
print('  Remember: public URL, no authentication. Stop it when you are done.')

### Try it without leaving the notebook

The call below posts a film straight to the local API — useful for checking the
response shape before opening the UI.

In [ ]:
import io, json, numpy as np, requests
from PIL import Image
from dvlhg.data.synthetic import render_cxr, make_report

rng = np.random.default_rng(1)
labels = np.array([0., 1., 0., 1.], np.float32)   # cardiomegaly + effusion
buffer = io.BytesIO()
Image.fromarray((render_cxr(labels, 256, rng) * 255).astype('uint8'), 'L').save(buffer, 'PNG')

response = requests.post('http://127.0.0.1:8000/api/predict',
                         files={'image': ('film.png', buffer.getvalue(), 'image/png')},
                         data={'report': make_report(labels, rng), 'explain': 'true'})
result = response.json()
for finding in result['findings']:
    flag = 'FLAG' if finding['flagged'] else '    '
    print(f"  {flag} {finding['label']:18s} {finding['probability']:.3f}"
          f"  (threshold {finding['threshold']:.2f}, without hypergraph {finding['without_hypergraph']:.3f})")
print('\nlatency', result['latency_ms'], 'ms | modality gate', result['modality_gate'])
print('neighbours:', [(n['similarity'], n['findings']) for n in result.get('neighbours', [])[:3]])

### Shut down

In [ ]:
tunnel.terminate(); server.terminate()
print('tunnel and server stopped')